In [1]:
import torch
print("CUDA Available: ", torch.cuda.is_available())
print("CUDA Device Name: ", torch.cuda.get_device_name(0))
torch.cuda.empty_cache()

# Verify CUDA
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using Device: {device}")

CUDA Available:  True
CUDA Device Name:  NVIDIA GeForce RTX 3050 Ti Laptop GPU
Using Device: cuda


In [1]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, SparseVectorParams
from app.utils.settings import QDRANT_HOST, QDRANT_PORT, COLLECTION_NAME, CHUNKS_FILE
from app.utils.chunking import load_chunks

/home/arimatea/Documents/Pessoal/Mestrado/Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_chatbotUI/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def ingest_chunks_to_qdrant():
    """
    Ingests pre-chunked documents into Qdrant vector store using local embeddings.
    Creates the collection if it doesn't exist and adds texts + metadata in batch.
    """
    # 1. Initialize local embeddings model
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": "cpu"},  # change to "cuda" if GPU is available
        encode_kwargs={"normalize_embeddings": True},
    )

    # 2. Connect to Qdrant instance
    client = QdrantClient(
        host=QDRANT_HOST,
        port=QDRANT_PORT,
        timeout=120,
    )

    # 3. Create collection if it doesn't exist (fixed dimension + named vectors)
    if not client.collection_exists(collection_name=COLLECTION_NAME):
        client.create_collection(
            collection_name=COLLECTION_NAME,
            vectors_config={
                "text-dense": VectorParams(size=384, distance=Distance.COSINE)
            },
            sparse_vectors_config={
                "text-sparse": SparseVectorParams()  # no size needed for sparse vectors
            },
        )
        print(f"Collection '{COLLECTION_NAME}' created with 'text-dense' vector.")
    else:
        print(f"Collection '{COLLECTION_NAME}' already exists.")

    # 4. Initialize LangChain Qdrant vector store with named vector
    vector_store = QdrantVectorStore(
        client=client,
        collection_name=COLLECTION_NAME,
        embedding=embeddings,
        vector_name="text-dense",          # must match the named vector created
        sparse_vector_name="text-sparse",  # optional, only if you plan to use hybrid search
        # force_recreate=True, # recreate collection (use with caution!)
    )

    # 5. Load chunks and prepare data for ingestion
    chunks = load_chunks(CHUNKS_FILE)
    if not chunks:
        print("No chunks found to ingest.")
        return

    texts = [chunk["content"] for chunk in chunks]
    metadatas = [
        {
            "release": chunk.get("release", ""),
            "series": chunk.get("series", ""),
            "spec": chunk.get("spec", ""),
            "text": chunk["content"]  # optional: store full text in payload if needed
        }
        for chunk in chunks
    ]

    # 6. Batch ingest texts + metadata into Qdrant (efficient and clean)
    vector_store.add_texts(
        texts=texts,
        metadatas=metadatas,
        # ids=[...]  # optional: provide custom IDs if you need deterministic behavior
    )

    print(f"Successfully ingested {len(texts)} chunks into Qdrant collection '{COLLECTION_NAME}'.")

In [23]:
ingest_chunks_to_qdrant()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 543.49it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Collection '3gpp_rel18_28' already exists.
Chunks loaded from ../../../files/tspec_chunks.pkl
Successfully ingested 1123 chunks into Qdrant collection '3gpp_rel18_28'.


# Test collection

In [29]:
client = QdrantClient(
        host=QDRANT_HOST,
        port=QDRANT_PORT,
        timeout=120
        )
collection_info = client.get_collection(COLLECTION_NAME)
print(collection_info)
print(f"\nCollection: {COLLECTION_NAME}")
print(f"Status: {collection_info.status}")
print(f"Points count: {collection_info.points_count:,}")

status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=2246 segments_count=8 config=CollectionConfig(params=CollectionParams(vectors={'text-dense': VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None)}, shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, on_disk_payload=True, sparse_vectors={'text-sparse': SparseVectorParams(index=None, modifier=None)}), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None